This project uses the Kaggle cancer histopathology dataset.

**Note:** The dataset contains metastatic breast cancer tissue in lymph nodes, not lung cancer.

In [ ]:

# نصب کتابخانه‌های مورد نیاز
!pip -q install kagglehub torchmetrics seaborn


PyTorch is used with `DataLoader` for batch-based loading.

In [ ]:

import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


### Kaggle

Upload `kaggle.json` if Kaggle authentication is required, then run the setup cell.

In [ ]:

# اگر Kaggle API در محیط شما فعال است، از این روش استفاده کنید.
# قبل از اجرای این سلول، فایل kaggle.json را در Colab تنظیم کنید.

# در Colab:
# from google.colab import files
# files.upload()  # فایل kaggle.json را انتخاب کنید

# سپس:
# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/kaggle.json
# !chmod 600 ~/.kaggle/kaggle.json
# !pip -q install kaggle
# !kaggle competitions download -c histopathologic-cancer-detection -p /content/data
# !unzip -q /content/data/histopathologic-cancer-detection.zip -d /content/data

# مسیر معمول:
DATASET_DIR = Path("/content/data")

print("DATASET_DIR =", DATASET_DIR)
print("در صورت استفاده از مسیر دیگری، مقدار DATASET_DIR را اصلاح کنید.")


The code searches for `train/` and `train_labels.csv` automatically.

In [ ]:

from pathlib import Path

def find_file(root, filename):
    matches = list(Path(root).rglob(filename))
    return matches[0] if matches else None

def find_dir(root, dirname):
    matches = [p for p in Path(root).rglob(dirname) if p.is_dir()]
    return matches[0] if matches else None

LABELS_PATH = find_file(DATASET_DIR, "train_labels.csv")
TRAIN_DIR = find_dir(DATASET_DIR, "train")

print("Labels:", LABELS_PATH)
print("Train directory:", TRAIN_DIR)

if LABELS_PATH is None or TRAIN_DIR is None:
    print("\nساختار پوشه پیدا نشد. مسیر DATASET_DIR را بررسی کنید.")


Labels: `0 = No Tumor`, `1 = Tumor`.

In [ ]:

labels_df = pd.read_csv(LABELS_PATH)

print(labels_df.head())
print("\nShape:", labels_df.shape)
print("\nClass distribution:")
print(labels_df["label"].value_counts())
print("\nClass ratio:")
print(labels_df["label"].value_counts(normalize=True))


In [ ]:

plt.figure(figsize=(6, 4))
sns.countplot(data=labels_df, x="label")
plt.title("Class Distribution")
plt.xlabel("Label")
plt.ylabel("Number of images")
plt.show()


Default subset: 14,000 images: 10,000 train, 2,000 validation, 2,000 test.

In [ ]:

TOTAL_SAMPLES = 14000
TRAIN_SIZE = 10000
VAL_SIZE = 2000
TEST_SIZE = 2000

assert TRAIN_SIZE + VAL_SIZE + TEST_SIZE == TOTAL_SAMPLES

if TOTAL_SAMPLES > len(labels_df):
    TOTAL_SAMPLES = len(labels_df)
    print("TOTAL_SAMPLES به تعداد موجود کاهش یافت:", TOTAL_SAMPLES)

subset_df, _ = train_test_split(
    labels_df,
    train_size=TOTAL_SAMPLES,
    stratify=labels_df["label"],
    random_state=SEED
)

train_df, temp_df = train_test_split(
    subset_df,
    train_size=TRAIN_SIZE,
    stratify=subset_df["label"],
    random_state=SEED
)

val_df, test_df = train_test_split(
    temp_df,
    train_size=VAL_SIZE,
    stratify=temp_df["label"],
    random_state=SEED
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("\nTrain distribution:")
print(train_df["label"].value_counts(normalize=True))

print("\nValidation distribution:")
print(val_df["label"].value_counts(normalize=True))

print("\nTest distribution:")
print(test_df["label"].value_counts(normalize=True))



## 6. Augmentation and Normalization

برای Train از augmentation استفاده می‌کنیم تا مدل فقط تصاویر آموزشی را حفظ نکند.

برای Validation و Test فقط تبدیل‌های لازم را انجام می‌دهیم.


In [ ]:

IMG_SIZE = 96
BATCH_SIZE = 64

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.15
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


Images are loaded on demand to reduce RAM usage.

In [ ]:

class HistopathologyDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_id = str(row["id"])
        label = float(row["label"])

        image_path = self.image_dir / f"{image_id}.tif"

        if not image_path.exists():
            # برای اطمینان در صورت وجود پسوند متفاوت
            candidates = list(self.image_dir.glob(f"{image_id}.*"))
            if not candidates:
                raise FileNotFoundError(f"Image not found: {image_id}")
            image_path = candidates[0]

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(label, dtype=torch.float32)

        return image, label


In [ ]:

train_dataset = HistopathologyDataset(train_df, TRAIN_DIR, train_transform)
val_dataset = HistopathologyDataset(val_df, TRAIN_DIR, eval_transform)
test_dataset = HistopathologyDataset(test_df, TRAIN_DIR, eval_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))


## 8. View Samples

In [ ]:

def denormalize(img):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img = img.cpu() * std + mean
    return torch.clamp(img, 0, 1)

images, labels = next(iter(train_loader))

fig, axes = plt.subplots(3, 4, figsize=(12, 9))

for ax, img, label in zip(axes.flat, images[:12], labels[:12]):
    img = denormalize(img).permute(1, 2, 0).numpy()
    ax.imshow(img)
    ax.set_title(f"Label: {int(label.item())}")
    ax.axis("off")

plt.tight_layout()
plt.show()


A compact CNN is trained from scratch for binary classification.

In [ ]:

class CancerCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x.squeeze(1)


model = CancerCNN().to(DEVICE)

print(model)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("\nTotal parameters:", total_params)
print("Trainable parameters:", trainable_params)


`BCEWithLogitsLoss` and AdamW are used.

In [ ]:

criterion = nn.BCEWithLogitsLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)



## 11. Training and Validation


In [ ]:

def train_one_epoch(model, loader, criterion, optimizer):
    model.train()

    running_loss = 0.0
    all_probs = []
    all_labels = []

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()

        logits = model(images)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        probs = torch.sigmoid(logits)

        running_loss += loss.item() * images.size(0)
        all_probs.extend(probs.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_auc = roc_auc_score(all_labels, all_probs)

    return epoch_loss, epoch_auc


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()

    running_loss = 0.0
    all_probs = []
    all_labels = []

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        logits = model(images)
        loss = criterion(logits, labels)

        probs = torch.sigmoid(logits)

        running_loss += loss.item() * images.size(0)
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_auc = roc_auc_score(all_labels, all_probs)

    return epoch_loss, epoch_auc


Start with 10 epochs. Increase if needed.

In [ ]:

NUM_EPOCHS = 10

history = {
    "train_loss": [],
    "val_loss": [],
    "train_auc": [],
    "val_auc": []
}

best_val_auc = -np.inf
best_model_path = "/content/best_cancer_cnn.pth"

for epoch in range(NUM_EPOCHS):
    train_loss, train_auc = train_one_epoch(
        model, train_loader, criterion, optimizer
    )

    val_loss, val_auc = evaluate(
        model, val_loader, criterion
    )

    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_auc"].append(train_auc)
    history["val_auc"].append(val_auc)

    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train AUC: {train_auc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val AUC: {val_auc:.4f} | "
        f"LR: {current_lr:.6f}"
    )

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save(model.state_dict(), best_model_path)
        print("  -> Best model saved.")

print("\nBest Validation AUC:", best_val_auc)
print("Model path:", best_model_path)


## 13. Training Curves

In [ ]:

epochs = range(1, NUM_EPOCHS + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs, history["train_loss"], marker="o", label="Train Loss")
plt.plot(epochs, history["val_loss"], marker="o", label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(epochs, history["train_auc"], marker="o", label="Train AUC")
plt.plot(epochs, history["val_auc"], marker="o", label="Validation AUC")
plt.xlabel("Epoch")
plt.ylabel("ROC-AUC")
plt.title("Training and Validation ROC-AUC")
plt.legend()
plt.grid(True)
plt.show()


Load the best validation checkpoint and evaluate it on the test set.

In [ ]:

model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))

@torch.no_grad()
def predict_loader(model, loader):
    model.eval()

    all_probs = []
    all_labels = []

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)

        logits = model(images)
        probs = torch.sigmoid(logits)

        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.numpy())

    return np.array(all_labels), np.array(all_probs)


test_labels, test_probs = predict_loader(model, test_loader)

test_preds = (test_probs >= 0.5).astype(int)

accuracy = accuracy_score(test_labels, test_preds)
precision = precision_score(test_labels, test_preds, zero_division=0)
recall = recall_score(test_labels, test_preds, zero_division=0)
f1 = f1_score(test_labels, test_preds, zero_division=0)
auc = roc_auc_score(test_labels, test_probs)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"ROC-AUC  : {auc:.4f}")



## 15. Classification Report


In [ ]:

print(
    classification_report(
        test_labels,
        test_preds,
        target_names=["No Tumor", "Tumor"],
        digits=4,
        zero_division=0
    )
)


## 16. Confusion Matrix

In [ ]:

cm = confusion_matrix(test_labels, test_preds)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["No Tumor", "Tumor"],
    yticklabels=["No Tumor", "Tumor"]
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()


ROC-AUC evaluates performance across classification thresholds.

In [ ]:

fpr, tpr, thresholds = roc_curve(test_labels, test_probs)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, label=f"ROC-AUC = {auc:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.grid(True)
plt.show()



## 18. Sample Predictions

در این قسمت چند تصویر از Test را نشان می‌دهیم و احتمال پیش‌بینی‌شده را روی تصویر می‌نویسیم.


In [ ]:

# برای نمایش تصاویر باید batchها را دوباره پیمایش کنیم
display_images = []
display_labels = []
display_probs = []

model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        images_device = images.to(DEVICE)
        probs = torch.sigmoid(model(images_device)).cpu().numpy()

        for img, label, prob in zip(images, labels.numpy(), probs):
            display_images.append(denormalize(img))
            display_labels.append(int(label))
            display_probs.append(float(prob))

        if len(display_images) >= 16:
            break

fig, axes = plt.subplots(4, 4, figsize=(12, 12))

for ax, img, label, prob in zip(
    axes.flat,
    display_images[:16],
    display_labels[:16],
    display_probs[:16]
):
    img = img.permute(1, 2, 0).numpy()
    pred = int(prob >= 0.5)

    ax.imshow(img)
    ax.set_title(
        f"Actual: {label} | Pred: {pred}\nProb: {prob:.2f}"
    )
    ax.axis("off")

plt.tight_layout()
plt.show()


Useful for project analysis.

In [ ]:

test_results = test_df.copy()
test_results["probability"] = test_probs
test_results["prediction"] = test_preds

false_positive = test_results[
    (test_results["label"] == 0) &
    (test_results["prediction"] == 1)
].copy()

false_negative = test_results[
    (test_results["label"] == 1) &
    (test_results["prediction"] == 0)
].copy()

print("False Positives:", len(false_positive))
print("False Negatives:", len(false_negative))

print("\nSample False Positives:")
display(false_positive.head())

print("\nSample False Negatives:")
display(false_negative.head())


Use this function only with compatible histopathology images.

In [ ]:

def predict_image(image_path, model, transform=eval_transform):
    model.eval()

    image = Image.open(image_path).convert("RGB")
    original = image.copy()

    tensor = transform(image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        logit = model(tensor)
        probability = torch.sigmoid(logit).item()

    prediction = int(probability >= 0.5)

    plt.figure(figsize=(5, 5))
    plt.imshow(original)
    plt.axis("off")
    plt.title(
        f"Prediction: {'Tumor' if prediction == 1 else 'No Tumor'}\n"
        f"Probability: {probability:.3f}"
    )
    plt.show()

    return prediction, probability

# مثال:
# predict_image("/content/example.tif", model)


Save the trained model weights and metadata.

In [ ]:

FINAL_MODEL_PATH = "/content/histopathologic_cancer_cnn.pth"

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "img_size": IMG_SIZE,
        "class_names": ["No Tumor", "Tumor"],
        "test_auc": float(auc),
    },
    FINAL_MODEL_PATH
)

print("Saved:", FINAL_MODEL_PATH)


`BCEWithLogitsLoss` and AdamW are used.